# 🚀 JalanCerdas AI — Retrain Model v2

**YOLOv8s (small)** — lebih akurat dari YOLOv8n

Run semua cell dari atas ke bawah (Shift+Enter)

⏱️ **Estimasi**: 30-50 menit di GPU T4

## 1️⃣ Install & Check GPU

In [ ]:
!pip install -q ultralytics kagglehub pyyaml

import torch
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({mem:.1f} GB)")
else:
    print("❌ NO GPU! Runtime → Change runtime type → T4 GPU")

## 2️⃣ Download Dataset

In [ ]:
import kagglehub, os

print("📥 Downloading dataset...")
path = kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan")
print(f"✅ Downloaded: {path}")

# Show actual structure
print("\n📂 Actual dataset structure:")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level <= 2:
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level <= 1:
            subindent = ' ' * 2 * (level + 1)
            for d in dirs[:5]:
                print(f"{subindent}{d}/")
            for f in files[:3]:
                print(f"{subindent}{f}")
            if len(files) > 3:
                print(f"{subindent}... +{len(files)-3} files")

## 3️⃣ Prepare Dataset (VOC → YOLO)

In [ ]:
!git clone https://github.com/Srjeff27/jalancerdas-ai.git 2>/dev/null || true

import xml.etree.ElementTree as ET
import shutil
from pathlib import Path

CLASS_NAMES = [
    "retak_memanjang",
    "pengelupasan_lapisan_permukaan",
    "lubang",
    "retak_kulit_buaya",
    "retak_blok",
    "retak_pinggir",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

source = Path(path)
dataset_dir = Path("jalancerdas-ai/ai-model/dataset")

if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

# Find all XML files recursively
all_xmls = list(source.rglob("*.xml"))
print(f"Found {len(all_xmls)} XML files total")

# Find all images recursively
all_imgs = list(source.rglob("*.jpg")) + list(source.rglob("*.jpeg")) + list(source.rglob("*.png"))
print(f"Found {len(all_imgs)} images total")

# Show sample paths
if all_xmls:
    print(f"\nSample XML: {all_xmls[0]}")
if all_imgs:
    print(f"Sample IMG: {all_imgs[0]}")

In [ ]:
# Smart dataset builder — finds images + XML by matching filenames
import re

split_map = {"train": "train", "valid": "val", "test": "test"}
total_imgs, total_lbls = 0, 0

for src_split, dst_split in split_map.items():
    src_dir = source / src_split
    if not src_dir.exists():
        print(f"  ⚠️  {src_split}/ not found, skipping")
        continue

    dst_img = dataset_dir / dst_split / "images"
    dst_lbl = dataset_dir / dst_split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    # Find ALL images in this split (recursively)
    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
        imgs.extend(src_dir.rglob(ext))

    # Find ALL XMLs in this split (recursively)
    xmls = {xml.stem: xml for xml in src_dir.rglob("*.xml")}

    n_img, n_lbl = 0, 0
    for img_path in imgs:
        # Find matching XML
        xml_path = xmls.get(img_path.stem)

        # Copy image
        shutil.copy2(img_path, dst_img / img_path.name)
        n_img += 1

        # Convert XML → YOLO if exists
        if xml_path and xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                w = int(root.find("size/width").text)
                h = int(root.find("size/height").text)

                lines = []
                for obj in root.findall("object"):
                    name = obj.find("name").text
                    if name not in CLASS_TO_ID:
                        continue
                    cls_id = CLASS_TO_ID[name]
                    bbox = obj.find("bndbox")
                    xmin = float(bbox.find("xmin").text)
                    ymin = float(bbox.find("ymin").text)
                    xmax = float(bbox.find("xmax").text)
                    ymax = float(bbox.find("ymax").text)
                    cx = ((xmin + xmax) / 2) / w
                    cy = ((ymin + ymax) / 2) / h
                    bw = (xmax - xmin) / w
                    bh = (ymax - ymin) / h
                    lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

                (dst_lbl / (img_path.stem + ".txt")).write_text("\n".join(lines))
                n_lbl += len(lines)
            except Exception as e:
                print(f"  ⚠️  XML parse error: {xml_path.name}: {e}")
        else:
            # No XML — write empty label (image-only)
            (dst_lbl / (img_path.stem + ".txt")).write_text("")

    total_imgs += n_img
    total_lbls += n_lbl
    print(f"  {dst_split}: {n_img} images, {n_lbl} labels")

print(f"\n✅ Total: {total_imgs} images, {total_lbls} labels")

# Verify
for split in ['train', 'val', 'test']:
    img_dir = dataset_dir / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        print(f"  {split}/images: {count} files ✅")

## 4️⃣ Fix data.yaml Path

In [ ]:
import yaml

data_yaml_path = "jalancerdas-ai/ai-model/configs/data.yaml"
dataset_abs = os.path.abspath("jalancerdas-ai/ai-model/dataset")

with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = dataset_abs

with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"✅ data.yaml: path={data['path']}")
print(f"   nc={data['nc']}, names={list(data['names'].values())}")

## 5️⃣ Train YOLOv8s (200 epochs)

⏱️ 30-50 menit — **JANGAN tutup tab!**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

print("🚀 Training YOLOv8s...")
print("   30-50 minutes on GPU T4")
print("   Do NOT close this tab!\n")

results = model.train(
    data=data_yaml_path,
    epochs=200,
    batch=16,
    imgsz=640,
    device=0,
    patience=50,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    copy_paste=0.1,
    close_mosaic=15,
    project="jalancerdas-ai/ai-model/runs",
    name="train_v2",
    exist_ok=True,
)

best_pt = "jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt"
print(f"\n✅ Training done! Best: {best_pt}")

## 6️⃣ Evaluate

In [ ]:
model = YOLO(best_pt)
results = model.val(data=data_yaml_path, device=0)

print("\n📊 RESULTS")
print("=" * 50)
print(f"  mAP@50:     {results.box.map50:.4f}")
print(f"  mAP@50-95:  {results.box.map:.4f}")
print(f"  Precision:  {results.box.mp:.4f}")
print(f"  Recall:     {results.box.mr:.4f}")
f1 = 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr + 1e-8)
print(f"  F1 Score:   {f1:.4f}")
print("\n  Per-class mAP@50:")
for i, ap in enumerate(results.box.ap50):
    print(f"    {results.names[i]:<40} {ap:.4f}")

## 7️⃣ Export TFLite

In [ ]:
model = YOLO(best_pt)
export_path = model.export(format="tflite", imgsz=640, half=True, simplify=True)

size_mb = os.path.getsize(export_path) / (1024*1024)
print(f"\n✅ TFLite: {export_path} ({size_mb:.2f} MB)")

## 8️⃣ Download

In [ ]:
from google.colab import files

print("📥 Download model files...")
print(f"\nCopy to project:")
print(f"  {export_path} → mobile/assets/models/pothole_yolo.tflite")

files.download(export_path)
files.download(best_pt)

print("\n✅ Done! Rebuild APK after copy.")